# DSE 220: Deep Learning — Worksheet 2 Coding Supplement
## Convolutional Neural Networks: From Intuition to Code

---

**Instructions**

This notebook has **4 coding questions** that directly mirror the hand calculations in the worksheet.
The goal is to verify your intuition against PyTorch's output and to understand how tensors flow through a CNN.

- Fill in every line marked `### YOUR CODE HERE ###`.
- Do **not** change any other lines.
- Run each cell after filling it in and compare the output with your worksheet answers.
- **Paste a screenshot of your output after each coding question into your Google Document for submission.**

**Useful References**
- [`torch.nn.Conv2d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [`torch.nn.MaxPool2d`](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)
- [`torch.nn.ReLU`](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
- [`torch.nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
- [`torch.nn.Flatten`](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)
- [PyTorch Tensor shapes (NCHW)](https://pytorch.org/docs/stable/tensor_attributes.html)

---

In [ ]:
# ── Run this cell first — imports everything you will need ────────────────────
import torch
import torch.nn as nn

print('PyTorch version:', torch.__version__)

PyTorch version: 2.10.0+cpu


---
## Coding Question 1 — Manual Convolution with Stride 2  *(mirrors Worksheet Q1)*

### Background
In **Worksheet Q1** you applied a `(3×3×2)` filter to a `(5×5×2)` input volume with **no padding** and **stride 2**.
The output was a `2×2` feature map.

Here you will replicate that exact computation in PyTorch.

### Key concepts
- PyTorch tensors are in **NCHW** format: `(batch, channels, height, width)`.
- `nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)` creates a learnable conv layer.
- We manually assign the worksheet filter weights so the output is deterministic.

> **Shape hint:** A `Conv2d` with 1 output channel and 2 input channels has weight shape
> `(out_channels, in_channels, kH, kW)` = `(1, 2, 3, 3)`.

In [ ]:
# ── Step 1: Define the input volume ──────────────────────────────────────────
# Shape must be (N, C, H, W) = (1, 2, 5, 5)

channel1 = torch.tensor([
    [ 0,  1,  1,  1,  1],
    [ 1,  1,  0, -1,  0],
    [ 1,  0, -1, -1,  1],
    [ 0, -1, -1,  0,  1],
    [ 1,  0, -1,  1,  0],
], dtype=torch.float32)

channel2 = torch.tensor([
    [ 1,  1,  1,  0,  0],
    [ 0, -1,  1,  1,  0],
    [ 0,  1, -1, -1,  1],
    [-1,  0, -1, -1,  1],
    [ 1,  1, -1,  0,  0],
], dtype=torch.float32)

# torch.stack combines channel1 and channel2 along a new axis (dim=0 → channel axis)
# .unsqueeze(0) adds the batch dimension
input_volume = torch.stack([channel1, channel2], dim=0).unsqueeze(0)
print('Input shape:', input_volume.shape)  # expected: torch.Size([1, 2, 5, 5])

Input shape: torch.Size([1, 2, 5, 5])


In [ ]:
# ── Step 2: Define the filter weights ────────────────────────────────────────
filter_ch1 = torch.tensor([
    [ 1,  1,  0],
    [ 1,  0, -1],
    [ 0, -1, -1],
], dtype=torch.float32)

filter_ch2 = torch.tensor([
    [ 0,  1,  1],
    [-1,  0,  1],
    [-1, -1,  0],
], dtype=torch.float32)

# Stack into shape (out_channels=1, in_channels=2, kH=3, kW=3)
filter_weights = torch.stack([filter_ch1, filter_ch2], dim=0).unsqueeze(0)
print('Filter weights shape:', filter_weights.shape)  # expected: torch.Size([1, 2, 3, 3])

Filter weights shape: torch.Size([1, 2, 3, 3])


In [ ]:
# ── Step 3: Create the Conv2d layer ──────────────────────────────────────────
#
# Fill in the arguments:
#   in_channels  — how many channels does the input volume have?
#   out_channels — how many filters are we applying? (1 filter → 1 output channel)
#   kernel_size  — what is the spatial size of the filter?
#   stride       — what stride was used in Worksheet Q1?
#   padding      — was any padding applied?
#   bias         — the worksheet assumes bias = 0; set to False.
#
# Reference: https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html

conv = nn.Conv2d(
    in_channels  = 2,
    out_channels = 1,
    kernel_size  = 3,
    stride       = 2,
    padding      = 0,
    bias         = False
)

# Assign the worksheet filter weights to this layer
with torch.no_grad():
    conv.weight = nn.Parameter(filter_weights)

print('Conv layer weight shape:', conv.weight.shape)  # expected: torch.Size([1, 2, 3, 3])

Conv layer weight shape: torch.Size([1, 2, 3, 3])


In [ ]:
# ── Step 4: Run the forward pass ─────────────────────────────────────────────
with torch.no_grad():
    output = conv(input_volume)

print('Output feature map shape:', output.shape)
# What shape do you expect in NCHW? Think: (1, 1, ?, ?)
# Use the formula floor((W - F + 2P) / S) + 1 to confirm.

print('\nOutput feature map values:')
print(output[0, 0])   # remove batch and channel dims for display

# ── CHECK: Do these values match your Worksheet Q1 hand calculation? ──────────

Output feature map shape: torch.Size([1, 1, 2, 2])

Output feature map values:
tensor([[ 5.,  3.],
        [ 1., -2.]])


In [ ]:
# ── Step 5: Confirm the output size formula ───────────────────────────────────
# Variables: W=5 (input), F=3 (filter), P=0 (padding), S=2 (stride)

W, F, P, S = 5, 3, 0, 2
expected_size = (W - F + 2 * P) // S + 1

print(f'Formula gives output size: {expected_size}')
print(f'Actual output H (from PyTorch): {output.shape[2]}')
print(f'Match: {expected_size == output.shape[2]}')

Formula gives output size: 2
Actual output H (from PyTorch): 2
Match: True


---
## Coding Question 2 — ReLU Activation  *(mirrors Worksheet Q4)*

### Background
In **Worksheet Q4** you applied the ReLU activation element-wise to a new $4\times4$ input patch.

Here you will do the same in PyTorch and observe:
1. That the output values match your hand calculation.
2. That the tensor **shape does not change** through ReLU.

> **Hint:** `nn.ReLU()` is instantiated just like any other layer and called on a tensor.  
> Reference: https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html

In [ ]:
# ── Step 1: Define the input patch from Worksheet Q4 ─────────────────────────
# The patch is a single 4×4 matrix (single channel, single batch item).
# Shape: (N=1, C=1, H=4, W=4)

patch_2d = torch.tensor([
    [ 3, -1,  2,  0],
    [-4,  5, -3,  1],
    [ 0, -2,  4, -5],
    [ 2,  1, -1,  3],
], dtype=torch.float32)

# Add batch and channel dimensions so the shape is (1, 1, 4, 4)
input_patch = patch_2d.unsqueeze(0).unsqueeze(0)

print('Input patch shape:', input_patch.shape)   # expected: torch.Size([1, 1, 4, 4])
print('Input values:\n', input_patch[0, 0])

Input patch shape: torch.Size([1, 1, 4, 4])
Input values:
 tensor([[ 3., -1.,  2.,  0.],
        [-4.,  5., -3.,  1.],
        [ 0., -2.,  4., -5.],
        [ 2.,  1., -1.,  3.]])


In [ ]:
# ── Step 2: Apply ReLU ───────────────────────────────────────────────────────
# Instantiate an nn.ReLU layer, then apply it to input_patch.

relu = nn.ReLU()   # instantiate nn.ReLU()
relu_output = relu(input_patch)

print('After ReLU shape:', relu_output.shape)   # shape should be unchanged
print('\nAfter ReLU values:')
print(relu_output[0, 0])

# ── CHECK: Compare with your Worksheet Q4 hand computation. ──────────────────

After ReLU shape: torch.Size([1, 1, 4, 4])

After ReLU values:
tensor([[3., 0., 2., 0.],
        [0., 5., 0., 1.],
        [0., 0., 4., 0.],
        [2., 1., 0., 3.]])


In [ ]:
# ── Step 3: Observe shape invariance ─────────────────────────────────────────
print('Shape before ReLU:', input_patch.shape)
print('Shape after  ReLU:', relu_output.shape)
print('Shapes are equal:', input_patch.shape == relu_output.shape)

Shape before ReLU: torch.Size([1, 1, 4, 4])
Shape after  ReLU: torch.Size([1, 1, 4, 4])
Shapes are equal: True


---
## Coding Question 3 — Max-Pooling with Different Settings  *(mirrors Worksheet Q6)*

### Background
In **Worksheet Q6** you applied two max-pooling operations to a new $3\times3$ input patch:

- **Q6.1** — `MaxPool2d(kernel=2×2, stride=1)` → $2\times2$ output
- **Q6.2** — `MaxPool2d(kernel=3×3, stride=1, padding=1)` → $3\times3$ output

Here you will implement both in PyTorch and verify your hand-computed values.

> **Key argument:** `nn.MaxPool2d(kernel_size, stride, padding)`  
> Reference: https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html

In [ ]:
# ── Step 1: Define the input patch from Worksheet Q6 ─────────────────────────
# Shape: (N=1, C=1, H=3, W=3)

inp_2d = torch.tensor([
    [ 2,  0, -1],
    [-1,  4,  2],
    [ 3, -2,  1],
], dtype=torch.float32)

# Add batch and channel dimensions → shape (1, 1, 3, 3)
input_q6 = inp_2d.unsqueeze(0).unsqueeze(0)

print('Input shape:', input_q6.shape)   # expected: torch.Size([1, 1, 3, 3])
print('Input values:\n', input_q6[0, 0])

Input shape: torch.Size([1, 1, 3, 3])
Input values:
 tensor([[ 2.,  0., -1.],
        [-1.,  4.,  2.],
        [ 3., -2.,  1.]])


In [ ]:
# ── Step 2: Q6.1 — MaxPool2d(kernel=2, stride=1) ─────────────────────────────
#
# Fill in the kernel_size and stride from Worksheet Q6.1.
# No padding was used in Q6.1.

maxpool_q61 = nn.MaxPool2d(
    kernel_size = 2,
    stride      = 1,
)

output_q61 = maxpool_q61(input_q6)

print('Q6.1 — After MaxPool2d(kernel=2, stride=1):')
print('Output shape:', output_q61.shape)   # expected: torch.Size([1, 1, 2, 2])
print('Output values:\n', output_q61[0, 0])

# ── CHECK: Compare with your Worksheet Q6.1 hand computation. ────────────────

Q6.1 — After MaxPool2d(kernel=2, stride=1):
Output shape: torch.Size([1, 1, 2, 2])
Output values:
 tensor([[4., 4.],
        [4., 4.]])


In [ ]:
# ── Step 3: Q6.2 — MaxPool2d(kernel=3, stride=1, padding=1) ──────────────────
#
# Fill in kernel_size, stride, and padding from Worksheet Q6.2.

maxpool_q62 = nn.MaxPool2d(
    kernel_size = 3,
    stride      = 1,
    padding     = 1,
)

output_q62 = maxpool_q62(input_q6)

print('Q6.2 — After MaxPool2d(kernel=2, stride=2, padding=1):')
print('Output shape:', output_q62.shape)   # expected: torch.Size([1, 1, 3, 3])
print('Output values:\n', output_q62[0, 0])

# ── CHECK: Compare with your Worksheet Q6.2 hand computation. ────────────────

Q6.2 — After MaxPool2d(kernel=2, stride=2, padding=1):
Output shape: torch.Size([1, 1, 3, 3])
Output values:
 tensor([[4., 4., 4.],
        [4., 4., 4.],
        [4., 4., 4.]])


---
## Coding Question 4 — Building CNN Architectures and Counting Parameters  *(mirrors Worksheet Q9)*

### Background
In **Worksheet Q9** you calculated the feature-map shapes and total parameter counts for two CNN architectures by hand.

Here you will build both architectures in PyTorch, run a shape trace, and confirm your worksheet answers.

### Architectures

| **Architecture A** | **Architecture B** |
|---|---|
| `input → conv1 → conv2 → maxpool1 → fc1` | `input → conv3 → maxpool2 → fc2` |
| conv1: 3×3 kernel, 3 out-channels | conv3: 5×5 kernel, 3 out-channels |
| conv2: 3×3 kernel, 3 out-channels | maxpool2: 3×3 kernel, stride 2 |
| maxpool1: 3×3 kernel, stride 2 | fc2 → 10 outputs |
| fc1 → 10 outputs | |

**Input:** $(N, 3, 32, 32)$ — batch of 32×32 RGB images. No padding, stride 1 for all convolutions.

> **Tip:** Use your worksheet Q9 answers to fill in the correct layer dimensions.  
> `nn.Flatten()` reshapes `(N, C, H, W)` → `(N, C×H×W)` before the linear layer.

In [ ]:
# ── Helper functions (do not modify) ─────────────────────────────────────────

def count_params(model):
    """Total number of learnable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def shape_trace(model, x):
    """Run x through each sub-module and print its output shape."""
    print(f'  Input              : {tuple(x.shape)}')
    with torch.no_grad():
        for name, layer in model.named_children():
            x = layer(x)
            print(f'  After {name:<12}: {tuple(x.shape)}')
    return x

In [ ]:
# ── Architecture A ────────────────────────────────────────────────────────────
#
# Fill in all ### YOUR CODE HERE ### using your Q9 answers.
# Pay attention to:
#   - in_channels of conv2  = out_channels of conv1
#   - in_features of fc1    = C × H × W after maxpool1 (use your Q9(d) answer)

class ArchitectureA(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels  = 3,   # RGB input
            out_channels = 3,
            kernel_size  = 3,
            stride=1, padding=0
        )
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv2d(
            in_channels  = 3,   # output channels from conv1
            out_channels = 3,
            kernel_size  = 3,
            stride=1, padding=0
        )
        self.relu2 = nn.ReLU()

        self.maxpool1 = nn.MaxPool2d(
            kernel_size = 3,
            stride      = 2
        )

        self.flatten = nn.Flatten()

        # Hint: in_features = C_out × H_pool × W_pool
        # You computed H_pool and W_pool in Worksheet Q9(d).
        self.fc1 = nn.Linear(
            in_features  = 507,
            out_features = 10
        )

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.maxpool1(x)
        x = self.flatten(x)
        x = self.fc1(x)
        return x


model_A = ArchitectureA()
dummy_input = torch.randn(4, 3, 32, 32)   # 4-image batch, 3-channel, 32×32

print('=== Architecture A — Shape Trace ===')
_ = shape_trace(model_A, dummy_input)
print(f'\nTotal parameters (Architecture A): {count_params(model_A)}')
print('Compare with your Worksheet Q9(f) answer.')

=== Architecture A — Shape Trace ===
  Input              : (4, 3, 32, 32)
  After conv1       : (4, 3, 30, 30)
  After relu1       : (4, 3, 30, 30)
  After conv2       : (4, 3, 28, 28)
  After relu2       : (4, 3, 28, 28)
  After maxpool1    : (4, 3, 13, 13)
  After flatten     : (4, 507)
  After fc1         : (4, 10)

Total parameters (Architecture A): 5248
Compare with your Worksheet Q9(f) answer.


In [ ]:
# ── Architecture B ────────────────────────────────────────────────────────────

class ArchitectureB(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv3 = nn.Conv2d(
            in_channels  = 3,
            out_channels = 3,
            kernel_size  = 5,   # 5×5 kernel
            stride=1, padding=0
        )
        self.relu3 = nn.ReLU()

        self.maxpool2 = nn.MaxPool2d(
            kernel_size = 3,
            stride      = 2
        )

        self.flatten = nn.Flatten()

        self.fc2 = nn.Linear(
            in_features  = 507,
            out_features = 10
        )

    def forward(self, x):
        x = self.relu3(self.conv3(x))
        x = self.maxpool2(x)
        x = self.flatten(x)
        x = self.fc2(x)
        return x


model_B = ArchitectureB()

print('=== Architecture B — Shape Trace ===')
_ = shape_trace(model_B, dummy_input)
print(f'\nTotal parameters (Architecture B): {count_params(model_B)}')
print('Compare with your Worksheet Q9(f) answer.')

=== Architecture B — Shape Trace ===
  Input              : (4, 3, 32, 32)
  After conv3       : (4, 3, 28, 28)
  After relu3       : (4, 3, 28, 28)
  After maxpool2    : (4, 3, 13, 13)
  After flatten     : (4, 507)
  After fc2         : (4, 10)

Total parameters (Architecture B): 5308
Compare with your Worksheet Q9(f) answer.


In [ ]:
# ── Layer-by-layer parameter breakdown ───────────────────────────────────────
# This cell breaks down each layer's count — cross-check with Q9(e) and Q9(f).

def param_breakdown(model, model_name):
    print(f'\n--- {model_name} ---')
    total = 0
    for name, module in model.named_modules():
        params = sum(p.numel() for p in module.parameters(recurse=False))
        if params > 0:
            print(f'  {name:<12}: {params:>8,} parameters')
            total += params
    print(f'  {"TOTAL":<12}: {total:>8,} parameters')

param_breakdown(model_A, 'Architecture A')
param_breakdown(model_B, 'Architecture B')


--- Architecture A ---
  conv1       :       84 parameters
  conv2       :       84 parameters
  fc1         :    5,080 parameters
  TOTAL       :    5,248 parameters

--- Architecture B ---
  conv3       :      228 parameters
  fc2         :    5,080 parameters
  TOTAL       :    5,308 parameters


In [ ]:
# ── Inspect weight tensor shapes (cross-check Q9(a)) ─────────────────────────
# Weight shape format: (C_out, C_in, kH, kW)

print('conv1 weight shape:', model_A.conv1.weight.shape)
print('conv1 bias shape  :', model_A.conv1.bias.shape)
print()
print('conv2 weight shape:', model_A.conv2.weight.shape)
print('conv2 bias shape  :', model_A.conv2.bias.shape)
print()
print('conv3 weight shape:', model_B.conv3.weight.shape)
print('conv3 bias shape  :', model_B.conv3.bias.shape)
print()
print('fc1 weight shape  :', model_A.fc1.weight.shape)   # (out_features, in_features)
print('fc1 bias shape    :', model_A.fc1.bias.shape)
print()
print('fc2 weight shape  :', model_B.fc2.weight.shape)
print('fc2 bias shape    :', model_B.fc2.bias.shape)

conv1 weight shape: torch.Size([3, 3, 3, 3])
conv1 bias shape  : torch.Size([3])

conv2 weight shape: torch.Size([3, 3, 3, 3])
conv2 bias shape  : torch.Size([3])

conv3 weight shape: torch.Size([3, 3, 5, 5])
conv3 bias shape  : torch.Size([3])

fc1 weight shape  : torch.Size([10, 507])
fc1 bias shape    : torch.Size([10])

fc2 weight shape  : torch.Size([10, 507])
fc2 bias shape    : torch.Size([10])



You have now coded the core mechanics of CNNs from scratch in PyTorch!